In [7]:
# ==========================================
# Step 1: Install and Import Libraries
# ==========================================
!pip install nltk scikit-learn pandas

import pandas as pd
import numpy as np
import nltk
import re
import string

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

nltk.download('stopwords')

# ==========================================
# Step 2: Upload Dataset
# ==========================================
from google.colab import files
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

print("\nDataset Preview:")
print(df.head())

print("\nDataset Columns:")
print(df.columns)

# ==========================================
# Step 3: Detect Review and Rating Columns
# ==========================================
review_col = None
rating_col = None

possible_review_cols = ['reviewText','review','text','Review','Text']
possible_rating_cols = ['overall','rating','score','Score','Rating']

for col in df.columns:
    if col in possible_review_cols:
        review_col = col
    if col in possible_rating_cols:
        rating_col = col

# If still None, select first text column
if review_col is None:
    review_col = df.select_dtypes(include=['object']).columns[0]

# If still None, select first numeric column
if rating_col is None:
    rating_col = df.select_dtypes(include=['int64','float64']).columns[0]

print("\nDetected Review Column:", review_col)
print("Detected Rating Column:", rating_col)

# ==========================================
# Step 4: Convert Rating to Sentiment
# ==========================================
df['sentiment'] = df[rating_col].apply(lambda x: 1 if x >= 3 else 0)

# ==========================================
# Step 5: Text Cleaning
# ==========================================
stop_words = set(stopwords.words('english'))

def clean_text(text):

    text = str(text).lower()

    # remove URL
    text = re.sub(r"http\S+", "", text)

    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # remove numbers
    text = re.sub(r'\d+', '', text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    return " ".join(words)

df['clean_review'] = df[review_col].apply(clean_text)

# ==========================================
# Step 6: Feature Extraction (TF-IDF)
# ==========================================
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['clean_review'])
y = df['sentiment']

# ==========================================
# Step 7: Train Test Split
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ==========================================
# Step 8: Train Model
# ==========================================
model = LogisticRegression()

model.fit(X_train, y_train)

# ==========================================
# Step 9: Evaluate Model
# ==========================================
y_pred = model.predict(X_test)

print("\nModel Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# ==========================================
# Step 10: Predict New Review
# ==========================================
def predict_sentiment(review):

    review = clean_text(review)
    vector = vectorizer.transform([review])

    prediction = model.predict(vector)[0]

    if prediction == 1:
        print("Positive Review 😀")
    else:
        print("Negative Review 😡")

# ==========================================
# Example Predictions
# ==========================================
predict_sentiment("This product is amazing and works perfectly")
predict_sentiment("Very bad quality and waste of money")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Saving Amazon-Product-Reviews-Sentiment-Analysis-in-Python-Dataset.csv to Amazon-Product-Reviews-Sentiment-Analysis-in-Python-Dataset (2).csv

Dataset Preview:
                                              Review  Sentiment
0  Fast shipping but this product is very cheaply...          1
1  This case takes so long to ship and it's not e...          1
2  Good for not droids. Not good for iPhones. You...          1
3  The cable was not compatible between my macboo...          1
4  The case is nice but did not have a glow light...          1

Dataset Columns:
Index(['Review', 'Sentiment'], dtype='object')

Detected Review Column: Review
Detected Rating Column: Sentiment

Model Accuracy: 0.8158

Classification Report:

              precision    recall  f1-score   support

           0       0.80      0.72      0.76      2021
           1       0.82      0.88      0.85      2979

    accuracy                           0.82      5000
   macro avg       0.81      0.80      0.81      5000
weig